In [1]:
import os
from ultralytics import YOLO
from pathlib import Path

In [ ]:
DATA_YAML = "/home/diaz_asian/StampDetector/data/processed/YOLO_CONFIGS/data.yaml"
MODEL = "yolo11n.pt"        # or yolo11s.pt if you want slightly better boxes
IMG_SIZE = 640              # ← changed from 640 → critical for small stamps!
EPOCHS = 300                # ← you can train longer because we only optimize box loss
BATCH = -1                  # auto-batch with 70% GPU memory usage (best practice 2025)
NAME = f"v11_nano"
PROJECT = "StampDetector"

CUSTOM_ARGS = {
    "data": DATA_YAML,
    "epochs": EPOCHS,
    "batch": BATCH,
    "imgsz": IMG_SIZE,          # 800 or 1024 → huge impact on small stamp precision
    "name": NAME,
    "project": PROJECT,
    "exist_ok": True,
    "device": 0,                # change to "" if no GPU

    # Optimizer (perfect for small datasets + precise boxes)
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 5,

    # Loss weights — this is what makes boxes extremely accurate
    "box": 15.0,      # heavily penalize bad coordinates
    "cls": 0.0,       # completely ignore classification (you have 1 class)
    "dfl": 2.0,       # helps regression precision

    # Early stopping & best model selection
    "patience": 50,               # 50–100 is enough when monitoring box_loss

    # Document-specific tricks
    "rect": True,                 # perfect for scanned documents
    "close_mosaic": 15,           # last 15 epochs without mosaic → clean boxes
    "cache": "disk",              # safer for high-res documents (640+)
    "amp": True,
    "augment": True,              # keeps test-time augmentation
    "plots": True,
    "save": True,
}

In [4]:
### initialize model / run training / save results

model = YOLO(MODEL)

print(f"Training {MODEL} on {Path(DATA_YAML).parent}")
print(f"Results will be saved to: runs/detect/{PROJECT}/{NAME}")

results = model.train(**CUSTOM_ARGS)

Training yolo11n.pt on ../data/processed/YOLO_CONFIGS
Results will be saved to: runs/detect/StampDetector/v11_nano
Ultralytics 8.3.228 🚀 Python-3.11.7 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 9902MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=15.0, cache=disk, cfg=None, classes=None, close_mosaic=15, cls=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/processed/YOLO_CONFIGS/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v1

Exception in thread Thread-16 (_pin_memory_loop):
Traceback (most recent call last):
  File "/software/ncbr/softrepo/python/anaconda3/2024.02/x86_64/para/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/software/ncbr/softrepo/python/anaconda3/2024.02/x86_64/para/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/home/diaz_asian/StampDetector/.venv/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/diaz_asian/StampDetector/.venv/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/software/ncbr/softrepo/python/anaconda3/2024.02/x86_64/para/lib/python3.11/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^

: 